# Анализ качества изображений

In [ ]:
!pip install facenet-pytorch --no-cache-dir --no-deps
!pip install lpips


## Качество отдельных изображений

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

class ArtifactDetector:
    def __init__(self):
        pass

    def detect_artifacts(self, image_path):
        """
        Обнаруживает артефакты на изображении
        """
        img = cv2.imread(image_path)
        if img is None:
            print(f"Ошибка: не удалось загрузить изображение {image_path}")
            return None

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        artifacts = {}

        edges = cv2.Canny(gray, 50, 150)
        edge_density = np.sum(edges > 0) / edges.size
        artifacts['edge_density'] = edge_density

        noise_score = np.var(gray) / (np.mean(gray) + 1e-8)
        artifacts['noise_score'] = noise_score

        blockiness = self.detect_block_artifacts(gray)
        artifacts['blockiness'] = blockiness

        color_anomaly = self.detect_color_anomalies(img)
        artifacts['color_anomaly'] = color_anomaly

        return artifacts

    def detect_block_artifacts(self, gray_img, block_size=8):
        """
        Обнаруживает блочные артефакты (JPEG)
        """
        h, w = gray_img.shape
        h_blocks = h // block_size
        w_blocks = w // block_size

        block_variances = []
        for i in range(h_blocks):
            for j in range(w_blocks):
                block = gray_img[i*block_size:(i+1)*block_size,
                                 j*block_size:(j+1)*block_size]
                if block.size > 0:
                    block_variances.append(np.var(block))

        if len(block_variances) == 0 or np.mean(block_variances) == 0:
            return 0

        return np.std(block_variances) / np.mean(block_variances)

    def detect_color_anomalies(self, img):
        """
        Обнаруживает аномалии в цветовом пространстве
        """
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

        hue_hist = cv2.calcHist([hsv], [0], None, [180], [0, 180])
        hue_hist = hue_hist / (hue_hist.sum() + 1e-8)

        entropy = -np.sum(hue_hist[hue_hist > 0] * np.log2(hue_hist[hue_hist > 0] + 1e-8))

        return entropy

def estimate_image_quality(generated_images_paths, real_image_path):
    detector = ArtifactDetector()

    print("-" * 50)
    print("Анализ реального изображения")
    print("-" * 50)

    real_artifacts = detector.detect_artifacts(real_image_path)
    if real_artifacts:
        print(f"Реальное изображение: {os.path.basename(real_image_path)}")
        for key, value in real_artifacts.items():
            print(f"  {key}: {value:.4f}")
        print()

    print("-" * 50)
    print("Анализ полученных изображений")
    print("-" * 50)

    generated_results = []
    for i, img_path in enumerate(generated_images_paths):
        artifacts = detector.detect_artifacts(img_path)
        if artifacts:
            generated_results.append({
                'image': os.path.basename(img_path),
                'edge_density': artifacts['edge_density'],
                'noise_score': artifacts['noise_score'],
                'blockiness': artifacts['blockiness'],
                'color_anomaly': artifacts['color_anomaly']
            })
            print(f"Изображение {i+1}: {os.path.basename(img_path)}")
            print(f"  edge_density: {artifacts['edge_density']:.4f}")
            print(f"  noise_score: {artifacts['noise_score']:.4f}")
            print(f"  blockiness: {artifacts['blockiness']:.4f}")
            print(f"  color_anomaly: {artifacts['color_anomaly']:.4f}")
            print()

    if generated_results:
        df_generated = pd.DataFrame(generated_results)

        if real_artifacts:
            real_row = pd.DataFrame([{
                'image': 'REAL (reference)',
                'edge_density': real_artifacts['edge_density'],
                'noise_score': real_artifacts['noise_score'],
                'blockiness': real_artifacts['blockiness'],
                'color_anomaly': real_artifacts['color_anomaly']
            }])
            df_comparison = pd.concat([real_row, df_generated], ignore_index=True)
        else:
            df_comparison = df_generated

        # Сравнение с реальным
        if real_artifacts:
            print("\n" + "-" * 50)
            print("Сравнение с реальным изображением")
            print("-" * 50)

            metrics = ['edge_density', 'noise_score', 'blockiness', 'color_anomaly']
            for metric in metrics:
                real_val = real_artifacts[metric]
                gen_mean = df_generated[metric].mean()
                gen_std = df_generated[metric].std()

                diff_percent = abs(gen_mean - real_val) / (real_val + 1e-8) * 100

                print(f"\n{metric}:")
                print(f"  Реальное: {real_val:.4f}")
                print(f"  Сгенерированные (среднее): {gen_mean:.4f} ± {gen_std:.4f}")
                print(f"  Разница: {diff_percent:.1f}%")


real_image_path = "/content/real.jpeg"

generated_images_paths = [
    "/content/1226_00133_.png",
    "/content/1226_00137_.png",
    "/content/1226_00141_.png",
    "/content/1226_00145_.png",
]
estimate_image_quality(generated_images_paths, real_image_path)

--------------------------------------------------
Анализ реального изображения
--------------------------------------------------
Реальное изображение: селфи.jpeg
  edge_density: 0.0145
  noise_score: 47.6773
  blockiness: 5.4188
  color_anomaly: 4.6060

--------------------------------------------------
Анализ полученных изображений
--------------------------------------------------
Изображение 1: person_1_step_2.png
  edge_density: 0.0127
  noise_score: 43.9131
  blockiness: 5.4194
  color_anomaly: 4.8359

Изображение 2: person_2_step_2.png
  edge_density: 0.0129
  noise_score: 44.3048
  blockiness: 5.3371
  color_anomaly: 4.8626

Изображение 3: person_3_step_2.png
  edge_density: 0.0165
  noise_score: 44.0146
  blockiness: 5.0368
  color_anomaly: 4.8669

Изображение 4: person_4_step_2.png
  edge_density: 0.0120
  noise_score: 45.1763
  blockiness: 5.4400
  color_anomaly: 4.7894

Изображение 5: person_5_step_2.png
  edge_density: 0.0126
  noise_score: 45.7999
  blockiness: 5.2870
  

## Сохранение идентичности


In [13]:
class SingleImageIdentityEvaluator:
    def __init__(self):
        from facenet_pytorch import InceptionResnetV1, MTCNN

        self.facenet = InceptionResnetV1(pretrained='vggface2').eval()
        self.mtcnn = MTCNN(image_size=160, keep_all=True)  # keep_all=True для детекции всех лиц

        self.transform = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def detect_all_faces(self, image_path):
        """
        Детектирует все лица на изображении
        Возвращает список (изображение лица, bounding box)
        """
        img = Image.open(image_path).convert('RGB')

        # Детектируем все лица
        boxes, probs = self.mtcnn.detect(img)

        if boxes is None:
            return []

        faces = []
        for i, box in enumerate(boxes):
            # Вырезаем лицо
            x1, y1, x2, y2 = map(int, box)
            face_img = img.crop((x1, y1, x2, y2))

            # Преобразуем для FaceNet
            face_tensor = self.transform(face_img).unsqueeze(0)

            # Получаем эмбеддинг
            with torch.no_grad():
                embedding = self.facenet(face_tensor).numpy()[0]
                embedding = embedding / np.linalg.norm(embedding)

            faces.append({
                'bbox': box,
                'embedding': embedding,
                'face_image': face_img,
                'confidence': probs[i] if probs is not None else 1.0
            })

        return faces

    def compare_faces_in_single_image(self, image_path, person1_bbox=None, person2_bbox=None):
        """
        Сравнивает лица на одном изображении

        Args:
            image_path: путь к изображению
            person1_bbox: координаты первого лица (опционально)
            person2_bbox: координаты второго лица (опционально)
        """
        # Детектируем все лица
        faces = self.detect_all_faces(image_path)

        if len(faces) < 2:
            return {
                'error': f'Найдено только {len(faces)} лиц, нужно минимум 2',
                'faces_detected': len(faces)
            }

        # Если координаты не указаны, берем первые два лица
        if person1_bbox is None:
            face1 = faces[0]
            face2 = faces[1]
        else:
            # Находим лица по координатам
            face1 = self._find_face_by_bbox(faces, person1_bbox)
            face2 = self._find_face_by_bbox(faces, person2_bbox)

        if face1 is None or face2 is None:
            return {'error': 'Лица по указанным координатам не найдены'}

        # Вычисляем сходство
        cosine_sim = np.dot(face1['embedding'], face2['embedding'])
        euclidean_dist = np.linalg.norm(face1['embedding'] - face2['embedding'])

        return {
            'cosine_similarity': float(cosine_sim),
            'euclidean_distance': float(euclidean_dist),
            'face1_bbox': face1['bbox'].tolist(),
            'face2_bbox': face2['bbox'].tolist(),
            'face1_confidence': float(face1['confidence']),
            'face2_confidence': float(face2['confidence'])
        }


if __name__ == "__main__":
    evaluator = SingleImageIdentityEvaluator()
    res = {
        'cosine_similarity': 0.0,
        'euclidean_distance': 0.0,
        'face1_confidence': [],
        'face2_confidence': []
    }
    images = [
        '/content/generated/person_1_step_2.png',
        '/content/generated/person_2_step_2.png',
        '/content/generated/person_3_step_2.png',
        '/content/generated/person_4_step_2.png',
        '/content/generated/person_5_step_2.png',
        '/content/generated/person_6_step_2.png'
    ]
    for img in images:
        result = evaluator.compare_faces_in_single_image(img)
        res['cosine_similarity'] += result['cosine_similarity']
        res['euclidean_distance'] += result['euclidean_distance']
        res['face1_confidence'].append(result['face1_confidence'])
        res['face2_confidence'].append(result['face2_confidence'])

    print("Результат сравнения лиц")
    print("-"*50)
    print(f"Косинусное сходство: {res['cosine_similarity']/6:.4f}")
    print(f"Евклидово расстояние: {res['euclidean_distance']/6:.4f}")
    print(f"face1_confidence: {res['face1_confidence']}")
    print(f"face2_confidence: {res['face2_confidence']}")
    result = evaluator.compare_faces_in_single_image('/content/селфи.jpeg')
    print(f"Косинусное сходство реального изображения: {res['cosine_similarity']:.4f}")
    print(f"Евклидово расстояние реального изображения: {res['euclidean_distance']:.4f}")



Результат сравнения лиц
--------------------------------------------------
Косинусное сходство: 0.4009
Евклидово расстояние: 1.0886
face1_confidence: [0.9999748468399048, 0.9998866319656372, 0.9998737573623657, 0.9997729659080505, 0.9999901056289673, 0.9999102354049683]
face2_confidence: [0.9999604225158691, 1.0, 0.9999573230743408, 0.9999998807907104, 0.9999027252197266, 0.9999597072601318]
Косинусное сходство реального изображения: 2.4057
Евклидово расстояние реального изображения: 6.5318


## Качество переноса лиц на шаблон


In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.models import inception_v3
from facenet_pytorch import InceptionResnetV1, MTCNN
from PIL import Image, ImageDraw
import numpy as np
import cv2
from skimage.feature import local_binary_pattern
import lpips
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.spatial.distance import cosine
import os

class FaceSwapQualityEvaluator:
    """
    Оценка качества подмены лиц на одном шаблоне
    Шаблон: человек с паспортом (2 лица: на селфи и на паспорте)
    """
    def __init__(self, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device

        self.mtcnn = MTCNN(image_size=160, keep_all=True, device=device)

        self.facenet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

        self.lpips_model = lpips.LPIPS(net='alex').to(device)

        self.face_transform = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

        self.lpips_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])

    def detect_faces_with_positions(self, image_path):
        """
        Детектирует все лица на изображении с их позициями
        Возвращает список лиц с метками (селфи/паспорт)
        """
        if not os.path.exists(image_path):
            print(f"Файл не найден: {image_path}")
            return []

        img = Image.open(image_path).convert('RGB')
        boxes, probs = self.mtcnn.detect(img)

        if boxes is None:
            print(f"Лица не обнаружены на {image_path}")
            return []

        faces = []
        img_width, img_height = img.size


        boxes_sorted = sorted(boxes, key=lambda x: x[0])

        for i, box in enumerate(boxes_sorted):
            x1, y1, x2, y2 = map(int, box)
            face_img = img.crop((x1, y1, x2, y2))

            face_tensor = self.face_transform(face_img).unsqueeze(0).to(self.device)
            with torch.no_grad():
                embedding = self.facenet(face_tensor).cpu().numpy()[0]
                embedding = embedding / np.linalg.norm(embedding)

            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2

            # левое лицо - паспорт, правое - селфи
            if i == 0:
                position = 'passport'
            else:
                position = 'selfie'

            faces.append({
                'bbox': box,
                'face_img': face_img,
                'embedding': embedding,
                'confidence': probs[i] if probs is not None else 1.0,
                'position': position,
                'center': (center_x, center_y)
            })

        return faces

    def get_face_by_position(self, image_path, target_position):
        """
        Получает лицо по позиции (selfie или passport)
        """
        faces = self.detect_faces_with_positions(image_path)

        for face in faces:
            if face['position'] == target_position:
                return face

        return None

    def _extract_face_embedding(self, image_path):
        """Извлекает эмбеддинг лица из изображения"""
        if not os.path.exists(image_path):
            print(f"Файл не найден: {image_path}")
            return None, None

        img = Image.open(image_path).convert('RGB')
        boxes, _ = self.mtcnn.detect(img)

        if boxes is None:
            print(f"Лицо не обнаружено на {image_path}")
            return None, None

        x1, y1, x2, y2 = map(int, boxes[0])
        face_img = img.crop((x1, y1, x2, y2))

        face_tensor = self.face_transform(face_img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            embedding = self.facenet(face_tensor).cpu().numpy()[0]
            embedding = embedding / np.linalg.norm(embedding)

        return face_img, embedding

    def calculate_facenet_similarity(self, source_face_path, target_image_path, target_position, age_adjusted=False):
        """
        Вычисляет косинусное сходство между исходным лицом и лицом на целевом изображении
        """
        if not os.path.exists(source_face_path):
            return {'error': f'Исходный файл не найден: {source_face_path}'}

        if not os.path.exists(target_image_path):
            return {'error': f'Целевой файл не найден: {target_image_path}'}

        source_face, source_emb = self._extract_face_embedding(source_face_path)

        if source_emb is None:
            return {'error': 'Исходное лицо не найдено'}

        target_face = self.get_face_by_position(target_image_path, target_position)

        if target_face is None:
            return {'error': f'Лицо на позиции {target_position} не найдено'}

        cosine_sim = np.dot(source_emb, target_face['embedding'])

        return {
            'metric': 'FaceNet Cosine Similarity',
            'value': float(cosine_sim),
            'position': target_position
        }

    def calculate_lpips(self, source_face_path, target_image_path, target_position):
        """
        Вычисляет LPIPS исходного лица и результата
        """
        source_face = Image.open(source_face_path).convert('RGB')

        target_face_obj = self.get_face_by_position(target_image_path, target_position)

        if target_face_obj is None:
            return {'error': f'Лицо на позиции {target_position} не найдено'}

        target_face = target_face_obj['face_img']

        source_face = source_face.resize((256, 256))
        target_face = target_face.resize((256, 256))

        source_tensor = self.lpips_transform(source_face).unsqueeze(0).to(self.device)
        target_tensor = self.lpips_transform(target_face).unsqueeze(0).to(self.device)

        with torch.no_grad():
            lpips_score = self.lpips_model(source_tensor, target_tensor).item()

        return {
            'metric': 'LPIPS',
            'value': float(lpips_score),
            'position': target_position,
            'note': 'Чем меньше значение, тем лучше'
        }


    def calculate_seamlessness(self, template_path, result_path, target_position):
        """
        Оценивает естественность вставки лица на границах с помощью LBP
        """
        if not os.path.exists(template_path):
            return {'error': f'Шаблон не найден: {template_path}'}

        if not os.path.exists(result_path):
            return {'error': f'Результат не найден: {result_path}'}

        template = cv2.imread(template_path, cv2.IMREAD_GRAYSCALE)
        result = cv2.imread(result_path, cv2.IMREAD_GRAYSCALE)

        if template is None or result is None:
            return {'error': 'Не удалось загрузить изображения'}

        if template.shape != result.shape:
            result = cv2.resize(result, (template.shape[1], template.shape[0]))

        faces_result = self.detect_faces_with_positions(result_path)
        target_face = None
        for face in faces_result:
            if face['position'] == target_position:
                target_face = face
                break

        if target_face is None:
            return {'error': f'Лицо на позиции {target_position} не найдено'}

        x1, y1, x2, y2 = map(int, target_face['bbox'])
        
        # Толщина границы ~5% от размера лица
        face_width = x2 - x1
        face_height = y2 - y1
        thickness = max(5, min(face_width, face_height) // 20)
        
        # Создаём маску границы
        boundary_mask = np.zeros(template.shape, dtype=np.uint8)
        cv2.rectangle(boundary_mask, (x1, y1), (x2, y2), 1, thickness)
        boundary_pixels = boundary_mask > 0
        
        if np.sum(boundary_pixels) == 0:
            texture_similarity = 0.5
        else:
            # LBP 
            radius = 3
            n_points = 8 * radius
            template_lbp = local_binary_pattern(template, n_points, radius, method='uniform')
            result_lbp = local_binary_pattern(result, n_points, radius, method='uniform')
            
            # Сравниваем гистограммы LBP на границе
            hist_template, _ = np.histogram(template_lbp[boundary_pixels], bins=59, range=(0, 59))
            hist_result, _ = np.histogram(result_lbp[boundary_pixels], bins=59, range=(0, 59))
            
            # Нормализуем
            hist_template = hist_template / (hist_template.sum() + 1e-8)
            hist_result = hist_result / (hist_result.sum() + 1e-8)
            
            # Пересечение гистограмм
            texture_similarity = np.sum(np.minimum(hist_template, hist_result))
            texture_similarity = np.clip(texture_similarity, 0, 1)
        
        return {
            'metric': 'Seamlessness Score',
            'value': float(texture_similarity),
            'position': target_position,
            'note': 'Чем выше значение, тем лучше (0-1) - основано на анализе текстуры LBP'
        }

    def calculate_color_consistency(self, result_path, target_position):
        """
        Оценивает согласованность цвета лица с фоном
        """
        if not os.path.exists(result_path):
            return {'error': f'Результат не найден: {result_path}'}

        img = cv2.imread(result_path)
        if img is None:
            return {'error': 'Не удалось загрузить изображение'}

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        faces = self.detect_faces_with_positions(result_path)
        target_face = None
        for face in faces:
            if face['position'] == target_position:
                target_face = face
                break

        if target_face is None:
            return {'error': f'Лицо на позиции {target_position} не найдено'}

        x1, y1, x2, y2 = map(int, target_face['bbox'])

        face_region = img_rgb[y1:y2, x1:x2]

        margin = 40
        h, w = img_rgb.shape[:2]

        bg_x1 = max(0, x1 - margin)
        bg_x2 = min(w, x2 + margin)
        bg_y1 = max(0, y1 - margin)
        bg_y2 = min(h, y2 + margin)

        bg_mask = np.ones((bg_y2 - bg_y1, bg_x2 - bg_x1), dtype=bool)
        face_in_bg = (y1 - bg_y1, y2 - bg_y1, x1 - bg_x1, x2 - bg_x1)
        bg_mask[face_in_bg[0]:face_in_bg[1], face_in_bg[2]:face_in_bg[3]] = False

        bg_region = img_rgb[bg_y1:bg_y2, bg_x1:bg_x2][bg_mask]

        if len(bg_region) == 0:
            consistency = 0.5
        else:
            face_mean = np.mean(face_region, axis=(0, 1))
            bg_mean = np.mean(bg_region, axis=0)
            color_diff = np.linalg.norm(face_mean - bg_mean) / 255

            face_hist = []
            bg_hist = []
            for c in range(3):
                face_hist.append(np.histogram(face_region[:,:,c], bins=32, density=True)[0])
                bg_hist.append(np.histogram(bg_region[:,c], bins=32, density=True)[0])

            hist_sim = np.mean([np.sum(np.minimum(f, b)) for f, b in zip(face_hist, bg_hist)])

            face_std = np.std(face_region, axis=(0, 1))
            bg_std = np.std(bg_region, axis=0)
            std_diff = np.linalg.norm(face_std - bg_std) / 255

            consistency = (
                0.4 * (1 - color_diff) +
                0.4 * hist_sim +
                0.2 * (1 - std_diff)
            )
            consistency = np.clip(consistency, 0, 1)

        return {
            'metric': 'Color Consistency',
            'value': float(consistency),
            'position': target_position,
            'note': 'Чем выше значение, тем лучше'
        }

    def evaluate_single_image(self, template_path, result_path, young_face_path, old_face_path):
        """
        Оценка качества для одного изображения, где заменены оба лица
        """
        print("\n" + "-"*60)
        print("Проверка файлов")
        print("-"*60)

        files_to_check = {
            'шаблон': template_path,
            'молодое лицо': young_face_path,
            'старое лицо': old_face_path,
            'результат': result_path
        }

        missing_files = []
        for name, path in files_to_check.items():
            if not os.path.exists(path):
                missing_files.append(f"{name}: {path}")
                print(f"Файл не найден: {name} -> {path}")
            else:
                print(f"Файл найден: {name} -> {path}")

        if missing_files:
            return {'error': f'Отсутствуют файлы: {missing_files}'}

        faces = self.detect_faces_with_positions(result_path)

        if len(faces) < 2:
            return {
                'error': f'Найдено только {len(faces)} лиц, ожидалось 2',
                'faces_detected': len(faces)
            }

        print(f"\nОбнаружено {len(faces)} лиц")
        for face in faces:
            print(f"   - {face['position']}: {face['bbox']}")

        selfie_face = None
        passport_face = None

        for face in faces:
            if face['position'] == 'selfie':
                selfie_face = face
            elif face['position'] == 'passport':
                passport_face = face

        results = {
            'selfie_swap': {},
            'passport_swap': {}
        }

        # Оценка для селфи
        if selfie_face:
            # FaceNet Cosine
            results['selfie_swap']['facenet'] = self.calculate_facenet_similarity(
                old_face_path, result_path, 'selfie', age_adjusted=False
            )

            # LPIPS
            results['selfie_swap']['lpips'] = self.calculate_lpips(
                old_face_path, result_path, 'selfie'
            )

            # Seamlessness
            results['selfie_swap']['seamlessness'] = self.calculate_seamlessness(
                template_path, result_path, 'selfie'
            )

            # Color Consistency
            results['selfie_swap']['color'] = self.calculate_color_consistency(
                result_path, 'selfie'
            )


            # Общая оценка
            grades = []
            for metric in ['facenet', 'lpips', 'seamlessness', 'color']:
                if metric in results['selfie_swap'] and 'grade' in results['selfie_swap'][metric]:
                    grades.append(results['selfie_swap'][metric]['grade'])

        # Оценка для паспорта
        if passport_face:
            # FaceNet
            results['passport_swap']['facenet'] = self.calculate_facenet_similarity(
                young_face_path, result_path, 'passport', age_adjusted=True
            )


            # LPIPS
            results['passport_swap']['lpips'] = self.calculate_lpips(
                young_face_path, result_path, 'passport'
            )

            # Seamlessness
            results['passport_swap']['seamlessness'] = self.calculate_seamlessness(
                template_path, result_path, 'passport'
            )

            # Color Consistency
            results['passport_swap']['color'] = self.calculate_color_consistency(
                result_path, 'passport'
            )

            # Общая оценка
            grades = []
            for metric in ['facenet', 'lpips', 'seamlessness', 'color']:
                if metric in results['passport_swap'] and 'grade' in results['passport_swap'][metric]:
                    grades.append(results['passport_swap'][metric]['grade'])


        return results


if __name__ == "__main__":
    evaluator = FaceSwapQualityEvaluator()

    template_path = "/content/селфи.jpeg"
    young_face_path = "/content/молодой.jpeg"
    old_face_path = "/content/старше.jpeg"
    result_path = "/content/person_5_step_2.png"
    results = evaluator.evaluate_single_image(
        template_path=template_path,
        result_path=result_path,
        young_face_path=young_face_path,
        old_face_path=old_face_path
    )

    print('\n')
    print("-" * 60)
    print("Качество переноса")
    print("-" * 60)

    if 'selfie_swap' in results and 'facenet' in results['selfie_swap']:
        print(f"\nЗАМЕНА СЕЛФИ:")
        if 'value' in results['selfie_swap']['facenet']:
            print(f"   FaceNet Cosine:  {results['selfie_swap']['facenet']['value']:.4f}")
        if 'value' in results['selfie_swap']['lpips']:
            print(f"   LPIPS:           {results['selfie_swap']['lpips']['value']:.4f}")
        if 'value' in results['selfie_swap']['seamlessness']:
            print(f"   Seamlessness:    {results['selfie_swap']['seamlessness']['value']:.4f}")
        if 'value' in results['selfie_swap']['color']:
            print(f"   Color:           {results['selfie_swap']['color']['value']:.4f}")

    if 'passport_swap' in results and 'facenet' in results['passport_swap']:
        print(f"\nЗАМЕНА ФОТО ДОКУМЕНТА:")
        if 'value' in results['passport_swap']['facenet']:
            print(f"   FaceNet Cosine:  {results['passport_swap']['facenet']['value']:.4f}")
        if 'value' in results['passport_swap']['lpips']:
            print(f"   LPIPS:           {results['passport_swap']['lpips']['value']:.4f}")
        if 'value' in results['passport_swap']['seamlessness']:
            print(f"   Seamlessness:    {results['passport_swap']['seamlessness']['value']:.4f}")
        if 'value' in results['passport_swap']['color']:
            print(f"   Color:           {results['passport_swap']['color']['value']:.4f}")


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth

------------------------------------------------------------
Проверка файлов
------------------------------------------------------------
Файл найден: шаблон -> /content/селфи.jpeg
Файл найден: молодое лицо -> /content/молодой.jpeg
Файл найден: старое лицо -> /content/старше.jpeg
Файл найден: результат -> /content/person_5_step_2.png

Обнаружено 2 лиц
   - passport: [338.0646667480469 544.571044921875 379.56463623046875 600.77783203125]
   - selfie: [683.7432861328125 188.30694580078125 975.802734375 576.20849609375]


------------------------------------------------------------
Качество переноса
------------------------------------------------------------

ЗАМЕНА СЕЛФИ:
   FaceNet Cosine:  0.7581
   LPIPS:           0.5997
   Seamlessness:    0.4300
   Color:           0.6455

ЗАМЕНА ФОТО ДОКУМЕНТА:
   FaceNet Cosine:  0.6673
